# Séance 6 — Agrégation, jointures et modèle en étoile

**Durée pratique : 3 h** &nbsp;·&nbsp; Ateliers 6.1 à 6.4

## Ce que vous saurez faire à la fin

- distinguer `agg`, `transform` et `apply` et choisir le bon ;
- contrôler la cardinalité d'une jointure et diagnostiquer une explosion de lignes ;
- passer du format large au format long et inversement ;
- livrer un modèle en étoile exploitable directement par Power BI ou Tableau.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  capteurs.csv
  clients.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


In [2]:
sales = load_dataset('ventes_transformees')
customers = pd.read_csv(RAW_DIR / 'clients.csv')
products = pd.read_csv(RAW_DIR / 'produits.csv')
stores = pd.read_csv(RAW_DIR / 'magasins.csv')

sales['date_commande'] = pd.to_datetime(sales['date_commande'], errors='coerce')

for label, frame in [('sales', sales), ('customers', customers),
                     ('products', products), ('stores', stores)]:
    print(f'{label:<10} {frame.shape}')

sales      (16992, 22)
customers  (3060, 5)
products   (39, 5)
stores     (9, 5)


---
## Atelier 6.1 — groupby : agg, transform, apply (50 min)

### Partie guidée

In [3]:
# agg : réduit chaque groupe à une ligne
by_category = sales.groupby('categorie').agg(
    nb_commandes=('id_commande', 'count'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    panier_median=('montant', 'median'),
    quantite_totale=('quantite', 'sum'),
).round(2).sort_values('ca_total', ascending=False)

by_category

,nb_commandes,ca_total,panier_moyen,panier_median,quantite_totale
categorie,,,,,
Informatique,5162,17445914.35,3379.68,1167.38,26602
Audio,3949,9234318.11,2338.39,1073.00,15078
Mobilier,3936,8363145.49,2124.78,717.48,18274
Accessoire,3870,7575740.71,1957.56,834.65,14978


Comparez `panier_moyen` et `panier_median`. Un écart important signale une distribution
asymétrique : quelques commandes très élevées tirent la moyenne vers le haut. Publier la
seule moyenne dans un tableau de bord donnerait une image fausse du panier typique.

In [4]:
# transform : renvoie un résultat ALIGNÉ sur les lignes d'origine
sales['ca_categorie'] = sales.groupby('categorie')['montant'].transform('sum')
sales['part_dans_categorie'] = (sales['montant'] / sales['ca_categorie'] * 100).round(4)

print(sales[['categorie', 'montant', 'ca_categorie', 'part_dans_categorie']].head())
print('\nSomme des parts par catégorie :')
print(sales.groupby('categorie')['part_dans_categorie'].sum().round(2))

      categorie  montant  ca_categorie  part_dans_categorie
0  Informatique   457.88   17445914.35               0.0026
1      Mobilier   140.96    8363145.49               0.0017
2         Audio  2073.36    9234318.11               0.0225
3  Informatique  2281.00   17445914.35               0.0131
4      Mobilier  1904.49    8363145.49               0.0228

Somme des parts par catégorie :
categorie
Accessoire      100.0
Audio           100.0
Informatique    100.0
Mobilier        100.0
Name: part_dans_categorie, dtype: float64


**La règle est simple.** `agg` réduit (une ligne par groupe), `transform` conserve
(autant de lignes qu'à l'entrée). Chaque fois que vous voulez comparer une valeur à la
statistique de son groupe, c'est `transform` qu'il vous faut, jamais un `merge` sur un
résultat de `agg`.

In [5]:
# apply : le plus souple, mais le plus lent. À réserver aux cas irréductibles.
def summarize_group(group):
    return pd.Series({
        'nb': len(group),
        'top_produit': group['id_produit'].mode().iloc[0] if len(group) else None,
        'part_annulees': (group['statut'] == 'annule').mean().round(3),
    })


sales.groupby('canal').apply(summarize_group, include_groups=False)

,nb,top_produit,part_annulees
canal,,,
boutique,5738,P0027,0.165
telephone,2865,P0015,0.166
web,8389,P0038,0.170


### Partie autonome

In [6]:
# Q1. Construire un tableau croisé : chiffre d'affaires par catégorie et par canal,
#     avec les totaux en marge. Utilisez pivot_table avec margins=True.
ca_pivot = sales.pivot_table(
    values='montant',
    index='categorie',
    columns='canal',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
    fill_value=0
)
display(ca_pivot.round(2))
# TODO

canal,boutique,telephone,web,Total
categorie,,,,
Accessoire,2168517.47,2311009.90,3096213.34,7575740.71
Audio,2826937.75,861985.02,5545395.34,9234318.11
Informatique,6269923.49,4832327.05,6343663.81,17445914.35
Mobilier,3379259.98,2469037.09,2514848.42,8363145.49
Total,14644638.69,10474359.06,17500120.91,42619118.66


In [7]:
# Q2. Pour chaque client, calculer : nombre de commandes, CA total, panier moyen,
#     date de première et de dernière commande, nombre de catégories distinctes achetées.
#     Puis identifier les dix meilleurs clients par CA.
clients_stats = sales.groupby('id_client').agg(
    nb_commandes=('id_commande', 'nunique'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    premiere_commande=('date_commande', 'min'),
    derniere_commande=('date_commande', 'max'),
    nb_categories=('categorie', 'nunique')
)

# Arrondir les montants et trier par chiffre d'affaires
clients_stats[['ca_total', 'panier_moyen']] = (
    clients_stats[['ca_total', 'panier_moyen']].round(2)
)

# Afficher les 10 meilleurs clients
top_10_clients = clients_stats.sort_values(
    'ca_total', ascending=False
).head(10)

print("=== Top 10 des clients par chiffre d'affaires ===")
display(top_10_clients)
# TODO

=== Top 10 des clients par chiffre d'affaires ===


,nb_commandes,ca_total,panier_moyen,premiere_commande,derniere_commande,nb_categories
id_client,,,,,,
C00696,7,1110180.18,158597.17,2023-02-12 00:00:00,2024-11-19 00:00:00,3
C02327,4,907834.48,226958.62,2023-10-31 00:00:00,2024-12-28 14:36:00,2
C00279,10,876999.45,87699.94,2023-01-24 00:00:00,2024-11-05 00:00:00,4
C02631,10,788266.88,78826.69,2023-01-12 00:00:00,2024-12-19 00:00:00,4
C00941,7,783179.31,111882.76,2023-01-29 00:00:00,2024-09-15 00:00:00,3
C00355,13,779167.48,59935.96,2023-02-12 11:09:00,2024-09-25 00:00:00,4
C02101,6,750743.89,125123.98,2023-05-01 11:00:00,2024-02-03 00:00:00,3
C00806,5,707386.06,141477.21,2023-03-05 00:00:00,2024-08-26 00:00:00,2
C02285,7,687164.50,98166.36,2023-05-19 00:00:00,2024-12-27 00:00:00,2


In [8]:
# Q3. Avec transform, ajouter une colonne signalant les commandes dont le montant
#     dépasse de plus de 50 % le montant médian de leur catégorie.
#     Quelle part des commandes cela représente-t-il ?
# 1. Calculer le montant médian de chaque catégorie avec transform
sales['mediane_categorie'] = (
    sales.groupby('categorie')['montant'].transform('median')
)

# 2. Signaler les commandes dépassant de plus de 50 % cette médiane
sales['commande_superieure_50pct'] = (
    sales['montant'] > sales['mediane_categorie'] * 1.5
)

# 3. Calculer la part de ces commandes
nb_commandes = sales['commande_superieure_50pct'].sum()
part_commandes = sales['commande_superieure_50pct'].mean() * 100

print("Nombre de commandes concernées :", nb_commandes)
print(f"Part des commandes concernées : {part_commandes:.2f} %")

# 4. Afficher quelques exemples
display(sales.loc[
    sales['commande_superieure_50pct'],
    ['id_commande', 'categorie', 'montant', 'mediane_categorie']
].head())

# Commentaire :
# Ces commandes ont un montant supérieur à 150 % du montant
# médian de leur catégorie.
# La part obtenue indique leur proportion dans le jeu de ventes.
# TODO

Nombre de commandes concernées : 5662
Part des commandes concernées : 33.32 %


,id_commande,categorie,montant,mediane_categorie
2,CMD0010050,Audio,2073.36,1073.000
3,CMD0001979,Informatique,2281.00,1167.385
4,CMD0010513,Mobilier,1904.49,717.480
8,CMD0009796,Accessoire,2570.70,834.650
10,CMD0010731,Informatique,1884.53,1167.385


---
## Atelier 6.2 — Jointures et explosion de lignes (55 min)

### Partie guidée

In [9]:
print('Lignes avant jointure :', len(sales))

with_products = sales.merge(products, on='id_produit', how='left', suffixes=('', '_prod'))
print('Après jointure produits :', len(with_products))

n_orphans = with_products['libelle'].isna().sum()
print(f'Lignes sans produit correspondant : {n_orphans}',
      f'({n_orphans / len(with_products):.2%})')

unknown_ids = sales.loc[~sales['id_produit'].isin(products['id_produit']), 'id_produit']
print('Identifiants fautifs :', unknown_ids.unique())

Lignes avant jointure : 16992
Après jointure produits : 16992
Lignes sans produit correspondant : 75 (0.44%)
Identifiants fautifs : <ArrowStringArray>
['P9999']
Length: 1, dtype: str


**Le réflexe à acquérir : compter les lignes avant et après chaque jointure.** Un `left join`
conserve le nombre de lignes *seulement si* la clé est unique à droite. Si elle ne l'est pas,
le résultat se multiplie sans avertissement.

In [10]:
# La table clients est saine : sa clé primaire est bien unique.
print('Identifiants clients dupliqués :', customers['id_client'].duplicated().sum())

# On simule donc le cas d'une table de droite dégradée — situation banale dès qu'une
# extraction est relancée deux fois ou qu'un export concatène deux périodes.
customers_with_duplicates = pd.concat([customers, customers.head(50)], ignore_index=True)
exploded = sales.merge(customers_with_duplicates, on='id_client', how='left')
print(f'{len(sales)} lignes -> {len(exploded)} après jointure sur une clé non unique')
print('Aucune erreur levée. Le problème ne se verra que bien plus loin.')

Identifiants clients dupliqués : 0
16992 lignes -> 17287 après jointure sur une clé non unique
Aucune erreur levée. Le problème ne se verra que bien plus loin.


In [11]:
# Le paramètre validate transforme le problème silencieux en erreur explicite
try:
    sales.merge(customers_with_duplicates, on='id_client', how='left', validate='many_to_one')
except Exception as error:
    print(type(error).__name__, ':', error)

MergeError : Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 id_client
   C00001
   C00002
   C00003
   C00004
   C00005 ...


Prenez l'habitude de passer `validate` sur **toutes** vos jointures. Le coût est nul et
l'erreur qu'il attrape est parmi les plus coûteuses à diagnostiquer plus tard.

### Partie autonome

In [ ]:
# Q4. Assembler les quatre tables en un seul DataFrame analytique :
#     sales + products + customers + stores.
#     Contraintes : vérifier le nombre de lignes après chaque étape, utiliser validate,
#     et traiter explicitement les clés orphelines (documentez votre choix).
# 1. Copier les ventes
sales_analytic = sales.copy()
n_initial = len(sales_analytic)

print("Lignes au départ :", n_initial)

# 2. Jointure avec les produits
sales_analytic = sales_analytic.merge(
    products,
    on='id_produit',
    how='left',
    validate='many_to_one',
    suffixes=('', '_produit'),
    indicator='_jointure_produit'
)

print("Après jointure produits :", len(sales_analytic))
print("Produits orphelins :",
      (sales_analytic['_jointure_produit'] == 'left_only').sum())

assert len(sales_analytic) == n_initial

# 3. Jointure avec les clients
sales_analytic = sales_analytic.merge(
    customers,
    on='id_client',
    how='left',
    validate='many_to_one',
    suffixes=('', '_client'),
    indicator='_jointure_client'
)

print("Après jointure clients :", len(sales_analytic))
print("Clients orphelins :",
      (sales_analytic['_jointure_client'] == 'left_only').sum())

assert len(sales_analytic) == n_initial

# 4. Jointure avec les magasins
sales_analytic = sales_analytic.merge(
    stores,
    on='id_magasin',
    how='left',
    validate='many_to_one',
    suffixes=('', '_magasin'),
    indicator='_jointure_magasin'
)

print("Après jointure magasins :", len(sales_analytic))
print("Magasins orphelins :",
      (sales_analytic['_jointure_magasin'] == 'left_only').sum())

assert len(sales_analytic) == n_initial

# 5. Repérer toutes les lignes avec une clé orpheline
sales_analytic['cle_orpheline'] = (
    (sales_analytic['_jointure_produit'] == 'left_only')
    | (sales_analytic['_jointure_client'] == 'left_only')
    | (sales_analytic['_jointure_magasin'] == 'left_only')
)

print("\nNombre total de lignes avec clé orpheline :",
      sales_analytic['cle_orpheline'].sum())

print("Nombre final de lignes :", len(sales_analytic))

display(sales_analytic.head())

# Choix de traitement des clés orphelines :
# On conserve les lignes grâce aux jointures LEFT.
# Cela évite de perdre des ventes et de fausser le CA total.
# Les lignes orphelines sont signalées pour correction ultérieure.
# Leurs informations de référentiel restent manquantes (NaN).
# TODO

Lignes au départ : 16992
Après jointure produits : 16992
Produits orphelins : 75
Après jointure clients : 16992
Clients orphelins : 0
Après jointure magasins : 16992
Magasins orphelins : 0

Nombre total de lignes avec clé orpheline : 75
Nombre final de lignes : 16992


,id_commande,date_commande,id_client,id_produit,id_magasin,quantite,prix_unitaire,remise_pct,canal,statut,ville_livraison,montant,libelle,categorie,sous_categorie,cout_achat,marge_unitaire,taux_marge,montant_apres_remise,panier_moyen_client,...,ca_categorie,part_dans_categorie,mediane_categorie,commande_superieure_50pct,libelle_produit,categorie_produit,sous_categorie_produit,cout_achat_produit,_jointure_produit,date_inscription,segment,ville,age,_jointure_client,nom_magasin,ville_magasin,region,surface_m2,_jointure_magasin,cle_orpheline
0,CMD0003112,2024-01-18 19:53:00,C02300,P0009,M05,3,160.66,5.0,telephone,retourne,lille,457.88,Clavier modele 3,Informatique,Clavier,86.41,74.25,0.4622,457.88,1478.870000,...,17445914.35,0.0026,1167.385,False,Clavier modele 3,Informatique,Clavier,86.41,both,2021-04-18,Particulier,PARIS,39.0,both,Boutique Bordeaux,Bordeaux,Nouvelle-Aquitaine,769,both,False
1,CMD0000107,2024-01-08 00:00:00,C02791,P0028,M03,2,70.48,0.0,boutique,livre,toulouse,140.96,Lampe modele 1,Mobilier,Lampe,34.81,35.67,0.5061,140.96,860.968571,...,8363145.49,0.0017,717.480,False,Lampe modele 1,Mobilier,Lampe,34.81,both,2024-02-19,Particulier,PARIS,36.0,both,Boutique Marseille,Marseille,PACA,460,both,False
2,CMD0010050,2023-04-15 00:00:00,990031,P0017,M04,3,691.12,0.0,boutique,livre,marseille,2073.36,Enceinte modele 2,Audio,Enceinte,347.41,343.71,0.4973,2073.36,1170.664000,...,9234318.11,0.0225,1073.000,True,Enceinte modele 2,Audio,Enceinte,347.41,both,2019-02-24,Professionnel,Bordeaux,NaN,both,Boutique Lille,Lille,Hauts-de-France,245,both,False
3,CMD0001979,2024-11-06 00:00:00,C01883,P0010,M04,4,570.25,0.0,web,retourne,marseille,2281.00,Souris modele 1,Informatique,Souris,283.64,286.61,0.5026,2281.00,1225.822500,...,17445914.35,0.0131,1167.385,True,Souris modele 1,Informatique,Souris,283.64,both,2019-06-01,Particulier,Nantes,77.0,both,Boutique Lille,Lille,Hauts-de-France,245,both,False
4,CMD0010513,2024-02-11 00:00:00,C00571,P0024,M02,2,1058.05,10.0,web,livre,marseille,1904.49,Bureau modele 3,Mobilier,Bureau,554.59,503.46,0.4758,1904.49,1392.776667,...,8363145.49,0.0228,717.480,True,Bureau modele 3,Mobilier,Bureau,554.59,both,2021-01-02,Particulier,Strasbourg,44.0,both,Boutique Lyon,Lyon,Auvergne-Rhone-Alpes,853,both,False


In [15]:
# Q5. Écrire une fonction safe_merge(left, right, **kwargs) qui effectue la jointure
#     et lève une exception si le nombre de lignes change alors qu'un 'left join'
#     était demandé. Elle doit aussi signaler le taux de clés non appariées.
def safe_merge(left, right, **kwargs):
    n_initial = len(left)

    # Ajouter un indicateur pour repérer les clés non appariées
    options = kwargs.copy()
    options['indicator'] = '_origine_jointure'

    # Effectuer la jointure
    result = left.merge(right, **options)

    # Vérifier le nombre de lignes pour une jointure LEFT
    if options.get('how', 'inner') == 'left':
        if len(result) != n_initial:
            raise ValueError(
                f"Jointure incorrecte : {n_initial} lignes avant, "
                f"{len(result)} lignes après."
            )

        # Calculer le taux de clés non appariées
        n_non_appariees = (
            result['_origine_jointure'] == 'left_only'
        ).sum()

        taux = n_non_appariees / n_initial * 100 if n_initial else 0

        print(f"Clés non appariées : {n_non_appariees}")
        print(f"Taux de clés non appariées : {taux:.2f} %")

    # Supprimer la colonne temporaire
    result = result.drop(columns=['_origine_jointure'])

    return result 

# Test avec les ventes et les produits
test_merge = safe_merge(
    sales,
    products,
    on='id_produit',
    how='left',
    validate='many_to_one'
)

print("Lignes avant :", len(sales))
print("Lignes après :", len(test_merge))

# TODO

Clés non appariées : 75
Taux de clés non appariées : 0.44 %
Lignes avant : 16992
Lignes après : 16992


---
## Atelier 6.3 — Formats long et large (35 min)

In [16]:
# Format large : une colonne par mois, lisible par un humain
wide = sales.pivot_table(index='categorie',
                         columns=sales['date_commande'].dt.month,
                         values='montant', aggfunc='sum').round(0)
wide.columns = [f'mois_{month:02d}' for month in wide.columns]
wide.head()

,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,377700.0,290306.0,340485.0,660904.0,528372.0,345662.0,341055.0,1839610.0,486776.0,1024218.0,896564.0,444090.0
Audio,1202719.0,662225.0,393741.0,1094554.0,445550.0,423529.0,590389.0,1805403.0,424471.0,411295.0,1132486.0,647955.0
Informatique,1803339.0,1589647.0,3416002.0,689139.0,1626278.0,639193.0,1839227.0,734560.0,1694766.0,702179.0,711393.0,2000191.0
Mobilier,905905.0,315378.0,306098.0,1129765.0,1343534.0,623024.0,935075.0,460363.0,782626.0,386601.0,706635.0,468141.0


In [17]:
# Format long : une ligne par observation, exploitable par une machine
long = wide.reset_index().melt(id_vars='categorie', var_name='mois', value_name='ca')
long.head()

,categorie,mois,ca
0,Accessoire,mois_01,377700.0
1,Audio,mois_01,1202719.0
2,Informatique,mois_01,1803339.0
3,Mobilier,mois_01,905905.0
4,Accessoire,mois_02,290306.0


**Lequel choisir ?** Le format long pour tout traitement programmatique et pour les outils
de BI, qui attendent une ligne par fait. Le format large uniquement pour l'affichage final
destiné à un lecteur humain.

In [18]:
# Q6. Repasser du format long au format large sans utiliser pivot_table,
#     puis vérifier que vous retrouvez exactement le tableau de départ.
# 1. Repasser du format long au format large
wide_reconstructed = long.pivot(
    index='categorie',
    columns='mois',
    values='ca'
)

# 2. Remettre les colonnes dans le même ordre
wide_reconstructed = wide_reconstructed.reindex(columns=wide.columns)

# 3. Vérifier que les deux tableaux sont identiques
assert wide_reconstructed.equals(wide)

print("OK — Le tableau initial a été retrouvé !")

# 4. Afficher le résultat
display(wide_reconstructed)
# TODO

OK — Le tableau initial a été retrouvé !


,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,377700.0,290306.0,340485.0,660904.0,528372.0,345662.0,341055.0,1839610.0,486776.0,1024218.0,896564.0,444090.0
Audio,1202719.0,662225.0,393741.0,1094554.0,445550.0,423529.0,590389.0,1805403.0,424471.0,411295.0,1132486.0,647955.0
Informatique,1803339.0,1589647.0,3416002.0,689139.0,1626278.0,639193.0,1839227.0,734560.0,1694766.0,702179.0,711393.0,2000191.0
Mobilier,905905.0,315378.0,306098.0,1129765.0,1343534.0,623024.0,935075.0,460363.0,782626.0,386601.0,706635.0,468141.0


---
## Atelier 6.4 — Livrer un modèle en étoile (40 min)

Un outil de BI attend une **table de faits** (une ligne par événement mesurable, contenant
des clés étrangères et des mesures numériques) entourée de **tables de dimensions** (une ligne
par entité, contenant ses attributs descriptifs). Livrer une seule table plate oblige
l'utilisateur à refaire ce travail dans l'outil, souvent moins bien.

In [19]:
# Dimension temps : indispensable, souvent oubliée
full_range = pd.date_range(sales['date_commande'].min().normalize(),
                           sales['date_commande'].max().normalize(), freq='D')

dim_date = pd.DataFrame({'date': full_range})
dim_date['cle_date'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['annee'] = dim_date['date'].dt.year
dim_date['mois'] = dim_date['date'].dt.month
dim_date['nom_mois'] = dim_date['date'].dt.month_name()
dim_date['trimestre'] = dim_date['date'].dt.quarter
dim_date['semaine_iso'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['jour_semaine'] = dim_date['date'].dt.dayofweek
dim_date['nom_jour'] = dim_date['date'].dt.day_name()
dim_date['est_weekend'] = dim_date['jour_semaine'] >= 5

print(dim_date.shape)
dim_date.head(3)

(730, 10)


,date,cle_date,annee,mois,nom_mois,trimestre,semaine_iso,jour_semaine,nom_jour,est_weekend
0,2023-01-01,20230101,2023,1,January,1,52,6,Sunday,True
1,2023-01-02,20230102,2023,1,January,1,1,0,Monday,False
2,2023-01-03,20230103,2023,1,January,1,1,1,Tuesday,False


La dimension temps couvre **toute la plage en continu**, y compris les jours sans aucune
vente. Sans elle, un graphique d'évolution saute les jours vides et donne une impression
de continuité trompeuse.

In [20]:
# Q7. Construire le modèle complet et l'exporter en Parquet dans data/processed/star/ :
#     - fact_sales : cle_date, id_client, id_produit, id_magasin, quantite,
#       prix_unitaire, remise_pct, montant, montant_net, statut
#     - dim_date (déjà construite)
#     - dim_customer, dim_product, dim_store : dédoublonnées sur leur clé primaire
#     Vérifiez ensuite l'intégrité référentielle : toute clé de la table de faits
#     doit exister dans la dimension correspondante.
# 1. Créer le dossier d'export
STAR_DIR = PROCESSED_DIR / 'star'
STAR_DIR.mkdir(parents=True, exist_ok=True)

# 2. Construire les quatre dimensions
dim_customer = customers.drop_duplicates(subset='id_client').copy()
dim_product = products.drop_duplicates(subset='id_produit').copy()
dim_store = stores.drop_duplicates(subset='id_magasin').copy()
dim_date = dim_date.drop_duplicates(subset='cle_date').copy()

# 3. Construire la table de faits
fact_sales = sales.copy()

# Créer la clé de date au format AAAAMMJJ
fact_sales['cle_date'] = (
    pd.to_datetime(fact_sales['date_commande'], errors='coerce')
    .dt.strftime('%Y%m%d')
    .astype('Int64')
)

# Montant net : seules les ventes abouties sont comptabilisées
# Convention : une commande annulée ou retournée contribue 0 au CA net
fact_sales['montant_net'] = fact_sales['montant'].where(
    ~fact_sales['statut'].isin(['annule', 'retourne']),
    0
)

# Conserver uniquement les colonnes demandées
fact_sales = fact_sales[
    [
        'cle_date', 'id_client', 'id_produit', 'id_magasin',
        'quantite', 'prix_unitaire', 'remise_pct',
        'montant', 'montant_net', 'statut'
    ]
].copy()

# 4. Traiter les clés orphelines sans perdre de ventes
# On ajoute les identifiants manquants aux dimensions correspondantes.
# Leurs informations descriptives restent manquantes.

dimensions = {
    'cle_date': dim_date,
    'id_client': dim_customer,
    'id_produit': dim_product,
    'id_magasin': dim_store
}

for key, dimension in dimensions.items():

    # Repérer les clés présentes dans les faits mais absentes de la dimension
    missing_keys = pd.Index(
        fact_sales[key].dropna().unique()
    ).difference(dimension[key].dropna().unique())

    if len(missing_keys) > 0:
        print(f"{key} : {len(missing_keys)} identifiant(s) orphelin(s)")

        # Créer des lignes de dimension pour les clés inconnues
        unknown_rows = pd.DataFrame({key: missing_keys})

        dimension = pd.concat(
            [dimension, unknown_rows],
            ignore_index=True
        )

        if key == 'id_produit':
            dimension['libelle'] = dimension['libelle'].fillna(
                'Produit inconnu'
            )

        dimensions[key] = dimension

# Récupérer les dimensions complétées
dim_date = dimensions['cle_date']
dim_customer = dimensions['id_client']
dim_product = dimensions['id_produit']
dim_store = dimensions['id_magasin']

# 5. Vérifier l'intégrité référentielle
for key, dimension in dimensions.items():

    # Chaque clé de la table de faits doit exister dans sa dimension
    invalides = (
        fact_sales[key].isna()
        | ~fact_sales[key].isin(dimension[key])
    )

    nb_invalides = int(invalides.sum())

    print(f"{key} : {nb_invalides} clé(s) invalide(s)")

    assert nb_invalides == 0, (
        f"Erreur d'intégrité référentielle sur {key}"
    )

    # Vérifier également l'unicité des clés dans la dimension
    assert dimension[key].is_unique, (
        f"Clés dupliquées dans la dimension {key}"
    )

print("\nOK — Intégrité référentielle vérifiée !")

# 6. Exporter les cinq tables au format Parquet
tables = {
    'fact_sales': fact_sales,
    'dim_date': dim_date,
    'dim_customer': dim_customer,
    'dim_product': dim_product,
    'dim_store': dim_store
}

for name, table in tables.items():
    path = STAR_DIR / f'{name}.parquet'
    table.to_parquet(path, index=False)
    print(f"{name}.parquet : {len(table)} lignes enregistrées")

print("\nModèle en étoile exporté dans :", STAR_DIR)
# TODO

id_produit : 1 identifiant(s) orphelin(s)
cle_date : 0 clé(s) invalide(s)
id_client : 0 clé(s) invalide(s)
id_produit : 0 clé(s) invalide(s)
id_magasin : 0 clé(s) invalide(s)

OK — Intégrité référentielle vérifiée !
fact_sales.parquet : 16992 lignes enregistrées
dim_date.parquet : 730 lignes enregistrées
dim_customer.parquet : 3060 lignes enregistrées
dim_product.parquet : 40 lignes enregistrées
dim_store.parquet : 9 lignes enregistrées

Modèle en étoile exporté dans : c:\Users\flore\Desktop\cours\Master_Coda\préparation_manipulation_donnees\data\processed\star


In [22]:
# Q8. Écrire un contrôle automatique de l'intégrité référentielle :
#     pour chaque couple (table de faits, dimension), afficher le nombre de clés orphelines.
#     Le contrôle doit échouer bruyamment s'il en trouve.
def check_referential_integrity(fact_table, dimensions):
    """dimensions : dict {nom_colonne_cle: dataframe_dimension}"""

    for key, dimension in dimensions.items():

        # Repérer les clés absentes de la dimension
        orphan_mask = (
            fact_table[key].isna()
            | ~fact_table[key].isin(dimension[key])
        )

        # Compter les clés orphelines
        nb_orphelines = int(orphan_mask.sum())

        print(f"{key} : {nb_orphelines} clé(s) orpheline(s)")

        # Déclencher une erreur si des clés sont orphelines
        if nb_orphelines > 0:
            raise ValueError(
                f"Erreur d'intégrité référentielle : "
                f"{nb_orphelines} clé(s) orpheline(s) pour {key}"
            )

    print("\nOK — Intégrité référentielle vérifiée !")
    
    dimensions = {
    'cle_date': dim_date,
    'id_client': dim_customer,
    'id_produit': dim_product,
    'id_magasin': dim_store
}

check_referential_integrity(fact_sales, dimensions)# TODO

cle_date : 0 clé(s) orpheline(s)
id_client : 0 clé(s) orpheline(s)
id_produit : 0 clé(s) orpheline(s)
id_magasin : 0 clé(s) orpheline(s)

OK — Intégrité référentielle vérifiée !


---
## Livrable de la séance

- `data/processed/star/` contenant la table de faits et les quatre dimensions en Parquet ;
- les fonctions `safe_merge` et `check_referential_integrity`, dans `src/` ;
- un schéma (même dessiné à la main et photographié) du modèle en étoile obtenu ;
- le compte des lignes à chaque étape de l'assemblage, avec explication des écarts.

**Ce livrable est le cœur de votre projet.** C'est le fichier que vous ouvrirez dans
Power BI ou Tableau, et celui dont la structure sera discutée en soutenance.